In [ ]:
import cv2
import numpy as np
import pandas as pd
import sys
import os
import time

sys.path.insert(0, os.path.abspath(".."))  # repo root, so `src` imports work


In [ ]:
from src.cryptosystem import Cryptosystem, Encryption, Decryption
from src.analysis.metrics import experimentalAnalysis
from src.analysis.differential import npcr,uaci

In [ ]:
time_file = '../outputs/time_complexity.csv'
time_records=[]
if os.path.exists(time_file):
    time_df = pd.read_csv(time_file)
else:
    # If file doesn't exist, create an empty DataFrame
    time_df = pd.DataFrame()

In [ ]:
image = cv2.imread('../images/fruits.jpg')
image_name = "fruits"

In [ ]:
x = Cryptosystem(image)

In [ ]:
key="101011111100110101000001010010110101101101010010100101010100011011001110110110010101111110010110000100010101000011000101100011011011111110011001111110101111111101001001111100111010101110000010000011101100100110000100000111101000100010101101"

In [ ]:
a1,b1,x0,y0 = x.generateInitials(key)
print(a1,b1,x0,y0)

In [ ]:
map1 = x.getChaoticMap(a1,b1,x0,y0)
map2 = x.getChaoticMap(a1,b1,map1[len(map1)-1][0],map1[len(map1)-1][1]) # map2 x0,y0 from the last values of map1 as given in paper
map3 = x.getChaoticMap(a1,b1,map2[len(map2)-1][0],map2[len(map2)-1][1])
map4 = x.getChaoticMap(a1,b1,map3[len(map3)-1][0],map3[len(map3)-1][1])
map5 = x.getChaoticMap(a1,b1,map4[len(map4)-1][0],map4[len(map4)-1][1])
map6 = x.getChaoticMap(a1,b1,map5[len(map5)-1][0],map5[len(map5)-1][1])


In [ ]:
rm1 = x.getRuleMap(map2,8)
rm2 = x.getRuleMap(map3,8)
rm3 = x.getRuleMap(map5,8)
rm4 = x.getRuleMap(map6,8)

In [ ]:
dsm = x.getDSMMap(map4,rm2)

In [ ]:
IVf= x.getfloorIV(8,map1)

In [ ]:
def save_inter_image(image_name,step,img="",rgb_flag=False,r="",g="",b=""):
    directory = f"../outputs/{image_name}/intermediate_images"
    os.makedirs(directory, exist_ok=True)  # Ensure directory exists

    file_path = os.path.join(directory, f"{image_name}_{step}.png")
    if rgb_flag:
        img = cv2.merge((b, g, r))
    success = cv2.imwrite(file_path, img)
    if success:
        print(f"{step} Image saved successfully at {file_path}")
    else:
        print(f"Failed to save {step} image. Check if the image is valid.")

In [ ]:
e = Encryption(image)
r,g,b = e.matrices()

### 1. intermix

In [ ]:
start_time = time.time()
rd = e.InterMix(r,g,b)
gd = e.InterMix(g,rd,b)
bd = e.InterMix(b,rd,gd)
end_time = time.time()
time_records.append(round(end_time - start_time, 2))
# save_inter_image(image_name,step="intermix", rgb_flag=True, r=rd, g=gd, b=bd)

### 1.2 Mixrows

In [ ]:
start_time = time.time()
mr,mg,mb = e.mixRows(rd,gd,bd,IVf)
end_time = time.time()
time_records.append(round(end_time - start_time, 2))
# save_inter_image(image_name, step="mixrows", rgb_flag=True, r=mr, g=mg, b=mb)

### 2. ACM

In [ ]:
start_time = time.time()
acmR,acmG,acmB = e.applyACM_numba(mr,mg,mb)
end_time = time.time()
time_records.append(round(end_time - start_time, 2))
# save_inter_image(image_name, step="acm", rgb_flag=True, r=acmR, g=acmG, b=acmB)

### 3. Spiral Mixing

In [ ]:
start_time = time.time()
smr = e.SpiralMixing(acmR)
smg = e.SpiralMixing(acmG)
smb = e.SpiralMixing(acmB)
end_time = time.time()
time_records.append(round(end_time - start_time, 2))
# save_inter_image(image_name, step="spiralmixing", rgb_flag=True, r=smr, g=smg, b=smb)

### 4. Encoding

In [ ]:
start_time = time.time()
ecdR = e.Encoding(smr,rm1)
ecdG = e.Encoding(smg,rm1)
ecdB = e.Encoding(smb,rm1)
end_time = time.time()
time_records.append(round(end_time - start_time, 2))

### 5. Substitution

In [ ]:
start_time = time.time()
subR = e.Substitution(ecdR,dsm,rm3)
subG = e.Substitution(ecdG,dsm,rm3)
subB = e.Substitution(ecdB,dsm,rm3)
end_time = time.time()
time_records.append(round(end_time - start_time, 2))

### 6. Decoding

In [ ]:
start_time = time.time()
decR = e.Decoding(subR,rm4)
decG = e.Decoding(subG,rm4)
decB = e.Decoding(subB,rm4)
end_time = time.time()
time_records.append(round(end_time - start_time, 2))

In [ ]:
cr = e.toMatrix(decR)
cg = e.toMatrix(decG)
cb = e.toMatrix(decB)

## final cipher image

In [ ]:
cipher = e.getFinalCipherImage(cr,cg,cb)

In [ ]:
directory = f"../outputs/{image_name}"
os.makedirs(directory, exist_ok=True)  # Ensure directory exists

file_path = os.path.join(directory, f"{image_name}_cipher.png")
success = cv2.imwrite(file_path, cipher)
if success:
    print(f"Image saved successfully at {file_path}")
else:
    print("Failed to save image. Check if the image is valid.")

# Decryption


In [ ]:
d = Decryption(cipher)

In [ ]:
dr,dg,db = d.matrices()

### 1. Reverse Encoding

In [ ]:
dencr,dencg,dencb = d.encoding(dr,dg,db,rm4)

### 2. Reverse Substitution

In [ ]:
dsubr,dsubg,dsubb = d.revSubstitution(dencr,dencg,dencb,dsm,rm3)

### 3. Reverse Decoding

In [ ]:
ddecr,ddecg,ddecb = d.decoding(dsubr,dsubg,dsubb,rm1)

In [ ]:
ddecr = d.toMatrix(ddecr)
ddecg = d.toMatrix(ddecg)
ddecb = d.toMatrix(ddecb)

### 4. Reverse Spiral Mixing

In [ ]:
dsmr,dsmg,dsmb = d.revSpiralMixing(ddecr,ddecg,ddecb)

### 5. Reverse ACM

In [ ]:
dacmr,dacmg,dacmb = d.applyRevACM_numba(dsmr,dsmg,dsmb)

### 6.1 Rev Mixrows

In [ ]:
rmrr = d.rmixRow(dacmr,IVf)
rmrg = d.rmixRow(dacmg,IVf)
rmrb = d.rmixRow(dacmb,IVf)

### 6. Reverse Intermix

In [ ]:
rrev,grev,brev = d.revInterMix(rmrr,rmrg,rmrb)

## final decoded image

In [ ]:
decodeCipher = d.getDecodedImage(rrev,grev,brev)

In [ ]:
directory = f"../outputs/{image_name}/{image_name}_decrypted.png"
cv2.imwrite(directory, decodeCipher)

### checking pixel difference in original,cipher and decoded image

In [ ]:
x.checkDiff(r,cr) #original red and cipher red
x.checkDiff(r,rrev) #original red and decoded red

# Experimental results and analysis

In [ ]:
directory = f"../outputs/{image_name}"
ex = experimentalAnalysis(cipher, directory)

## histogram analysis

In [ ]:
ex.plotHistogram(r,g,b,256, name="org_img_histogram")

In [ ]:
ex.plotHistogram(cr,cg,cb,256, name="cipher_img_histogram")

### variance

In [ ]:
vr = ex.variance_numba(r)
vg = ex.variance_numba(g)
vb = ex.variance_numba(b)

vdr = ex.variance_numba(cr)
vdg = ex.variance_numba(cg)
vdb = ex.variance_numba(cb)

In [ ]:
print("var of red",vr)
print("var of green",vr)
print("var of blue",vr)
print("var of cipher red",vdr)
print("var of cipher green",vdg)
print("var of cipher blue",vdb)

### chi square

In [ ]:
print("Chi Square of Cipher Red  ",ex.chiSquare(cr))
print("Chi Square of Cipher Green",ex.chiSquare(cg))
print("Chi Square of Cipher Blue ",ex.chiSquare(cb))

### entropy


In [ ]:
eRed = ex.entropy(r)
eGreen = ex.entropy(g)
eBlue = ex.entropy(b)
cRed = ex.entropy(cr)
cGreen = ex.entropy(cg)
cBlue = ex.entropy(cb)

In [ ]:
import pandas as pd
columns = ["Original Red", "Original Blue", "Original Green","Cipher Red", "Cipher Blue", "Cipher Green"]
entropy = pd.DataFrame(columns=columns)
row = [eRed,eGreen,eBlue,cRed,cGreen,cBlue]

entropy = pd.concat([entropy, pd.DataFrame([row], columns=entropy.columns)], ignore_index=True)
entropy.index = ['Entropy of Lenna Image']
entropy

### correlation analysis

In [ ]:
print('horizontal')
print('=======red========\n',ex.corAnalysisH(r))
print('\n=======green========\n',ex.corAnalysisH(g))
print('\n=======blue========\n',ex.corAnalysisH(b))
print('\n')
print('=======cipher red========\n',ex.corAnalysisH(cr))
print('\n=======cipher green========\n',ex.corAnalysisH(cg))
print('\n=======cipher blue========\n',ex.corAnalysisH(cb))

In [ ]:
print('vertical')
print('=======red========\n',ex.corAnalysisV(r))
print('\n=======green========\n',ex.corAnalysisV(g))
print('\n=======blue========\n',ex.corAnalysisV(b))
print('\n')
print('=======cipher red========\n',ex.corAnalysisV(cr))
print('\n=======cipher green========\n',ex.corAnalysisV(cg))
print('\n=======cipher blue========\n',ex.corAnalysisV(cb))

In [ ]:
print("diagonal\n")
print('=======red========\n',ex.corAnalysisD(r))
print('\n=======green========\n',ex.corAnalysisD(g))
print('\n=======blue========\n',ex.corAnalysisD(b))
print('\n')
print('=======cipher red========\n',ex.corAnalysisD(cr))
print('\n=======cipher green========\n',ex.corAnalysisD(cg))
print('\n=======cipher blue========\n',ex.corAnalysisD(cb))

## NPCR and UACI

In [ ]:
br=r.copy()
bg=g.copy()
bb=b.copy()
print(br[128][128],bg[128][128],bb[128][128])


br[128][128] = 117-32
bg[128][128] = 86-64
bb[128][128] = 55-32
print(br[128][128],bg[128][128],bb[128][128])

In [ ]:
imaged = cv2.merge((bb,bg,br))
start_time = time.time()
cipherd = e.applyEncryption(imaged, IVf, rm1, rm3, rm4, dsm)
end_time = time.time()
time_records.append(round(end_time - start_time, 2))

In [ ]:
cbd,cgd,crd = cv2.split(cipherd)

In [ ]:
print("=====NPCR======")
print(npcr(cr,crd))
print(npcr(cg,cgd))
print(npcr(cb,cbd))

print("=====UACI=====")
print(uaci(cr,crd))
print(uaci(cg,cgd))
print(uaci(cb,cbd))

## PSNR

In [ ]:
am,an,p = image.shape
def noiceAttack(image):
    for i in range(0,int(am/2)):
        for j in range(0,int(an/2)):
            image[i][j] = 1
    return image

In [ ]:
pr = cr.copy()
pg = cg.copy()
pb = cb.copy()

pr = noiceAttack(pr)
pg = noiceAttack(pg)
pb = noiceAttack(pb)

In [ ]:
import matplotlib.pyplot as plt
prgb = np.dstack((pr,pg,pb))
plt.imshow(prgb)
plt.show()
Prgb = cv2.merge((pb,pg,pr))

In [ ]:
import os
psnr_directory = f"../outputs/occlusion/{image_name}"
os.makedirs(psnr_directory, exist_ok=True)  # Ensure directory exists

file_path = os.path.join(psnr_directory, f"{image_name}_noise_attack.png")
success = cv2.imwrite(file_path, Prgb)
if success:
    print(f"Image saved successfully at {file_path}")
else:
    print("Failed to save image. Check if the image is valid.")

In [ ]:
print("Applying Decryption Algorithm on provided image\n")
pb,pg,pr = cv2.split(Prgb)
print("Step 1: applying Encoding\n")
enR,enG,enB = d.encoding(pr,pg,pb,rm4)
print("Step 2: applying Reverse Substitution\n")
rsR,rsG,rsB = d.revSubstitution(enR,enG,enB,dsm,rm3)
print("Step 3: applying Decoding\n")
dcR,dcG,dcB = d.decoding(rsR,rsG,rsB,rm1)
fred = d.toMatrix(dcR)
fgreen = d.toMatrix(dcG)
fblue = d.toMatrix(dcB)
print("Step 4: applying reverse spiral mixing\n")
mixR,mixG,mixB = d.revSpiralMixing(fred,fgreen,fblue)
print("Step 5: applying reverse ACM algorithm\n")
acmR,acmG,acmB = d.applyRevACM_numba(mixR,mixG,mixB)
print("\nStep 6: applying reverse mixrows\n")
mrr,mrg,mrb = d.revmixRows(acmR,acmG,acmB,IVf)
print("\nStep 7: applying reverse Intermix\n")
mrR,mrG,mrB = d.revInterMix(mrr,mrg,mrb)
decodedImage = d.getDecodedImage(mrR,mrG,mrB)

In [ ]:
decodedImage = cv2.merge((mrB,mrG,mrR))
file_path = os.path.join(psnr_directory, f"{image_name}_decrypted.png")
success = cv2.imwrite(file_path, decodedImage)
if success:
    print("success saved at",file_path)
else:
    print("unable to save")

In [ ]:
# PSNR now lives in src.analysis.metrics (ex.calculate_PSNR)


In [ ]:
import warnings
warnings.filterwarnings("ignore")
print(ex.calculate_PSNR(r,mrR))
print(ex.calculate_PSNR(g,mrG))
print(ex.calculate_PSNR(b,mrB))

In [ ]:
step_names = [
    "intermix",
    "mix_rows",
    "ACM",
    "spiral_mixing",
    "encoding",
    "substitution",
    "decoding",
    "total_time",
]

if not time_df.empty and image_name in time_df["image"].values:
    print(f"Image '{image_name}' already exists. Skipping...")
else:
    # Create a new row
    size = image.shape[0]
    row = {"image": image_name, "size": size}
    if len(time_records)<10:
        time_records.append(round(sum(time_records), 2))
    for step_name, time_value in zip(step_names, time_records):
        row[step_name] = time_value
    # row["total_time_in_sec"] = round(sum(time_records),3)

    # Append and save
    df_time = pd.concat([time_df, pd.DataFrame([row])], ignore_index=True)
    df_time.to_csv('../outputs/time_complexity.csv', index=False)
    print(f"Saved timing for image '{image_name}'.")